# Part 1 – Neural Network Fundamentals and Training Behavior Analysis
## Task 1: Dataset Understanding

**Dataset:** [Customer Churn Neural Network Dataset](https://drive.google.com/drive/folders/1akV6po4Nrgkc3yQrJkzA6cJlV-wBvUYs?usp=sharing)

**Goal:** Predict whether a customer will churn (`churn = 1`) or be retained (`churn = 0`)

**Note:** The raw dataset is loaded as-is. No rows, columns, or values are modified.


## 0. Imports & Configuration

In [ ]:
# Standard data science imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Output directory for results
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Plot styling
plt.rcParams.update({
    "figure.facecolor": "#F8FAFC",
    "axes.facecolor":   "#F8FAFC",
    "axes.edgecolor":   "#E2E8F0",
    "axes.grid":        True,
    "grid.color":       "#E2E8F0",
    "grid.linewidth":   0.8,
    "font.family":      "DejaVu Sans",
    "figure.dpi":       130,
})

BLUE = "#2563EB"
RED  = "#DC2626"

print("All libraries imported successfully.")


## 1. Load the Dataset

We load the CSV using `pandas.read_csv()` which auto-infers column data types.
`customer_id` is an identifier column — it will be noted but not used as a predictive feature.


In [ ]:
# Load the dataset — no modifications made to the raw file
df = pd.read_csv("customer_churn_nn.csv")

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


## 2. Number of Rows and Columns

Understanding dataset size is the first step — it informs:
- Whether we need mini-batch training (large datasets)
- The risk of overfitting (small datasets)
- Whether class imbalance may be a concern


In [ ]:
n_rows, n_cols = df.shape

print("=" * 40)
print(f"  Rows    : {n_rows:,}")
print(f"  Columns : {n_cols}")
print(f"  Features: {n_cols - 1}  (excluding target)")
print("=" * 40)


## 3. Type of Input Features

We classify columns into four categories:
- **Numerical** (`int64` / `float64`) — fed directly to the network after standardisation
- **Categorical** (`object`) — require one-hot encoding before training
- **Binary** — already 0/1, no encoding needed
- **Identifier** — non-predictive, dropped before training


In [ ]:
# Column names and inferred data types
print("Column Data Types:")
print("-" * 45)
for col in df.columns:
    print(f"  {col:<35} {str(df[col].dtype):<10}")

# Classify features
TARGET_COL   = 'churn'
ID_COL       = 'customer_id'
cat_cols     = ['region', 'plan_type', 'contract_type', 'payment_method']
binary_cols  = ['autopay_enabled']
num_cols     = ['tenure_months', 'monthly_charges_inr', 'avg_login_days_per_month',
                'support_tickets_last_90_days', 'payment_delay_days', 'data_usage_gb',
                'satisfaction_score', 'last_complaint_days_ago', 'discount_percent', 'referral_count']

print(f"\nNumerical features  ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"Binary features       ({len(binary_cols)}): {binary_cols}")
print(f"Identifier (drop)     : {ID_COL}")
print(f"Target column         : {TARGET_COL}")


## 4. Target Variable Description

The target column `churn` tells us whether a customer left (1) or stayed (0).
This is a **binary classification** problem:
- Loss function → `BCEWithLogitsLoss`
- Output layer → 1 neuron with Sigmoid activation


In [ ]:
print("Target Variable: 'churn'")
print("-" * 40)
print(f"  Data type     : {df['churn'].dtype}")
print(f"  Unique values : {sorted(df['churn'].unique())}")
print(f"  Total rows    : {len(df):,}")
print()
vc = df['churn'].value_counts().sort_index()
for label, val in zip(["Retained (0)", "Churned (1)"], vc.values):
    print(f"  {label}: {val:>5,}  ({val/len(df)*100:.1f}%)")
print()
print(f"  Imbalance ratio: {vc[0]/vc[1]:.1f} : 1  (majority : minority)")
print()
print("  ⚠ SEVERE CLASS IMBALANCE detected.")
print("    A naive 'always predict retained' model scores 98.4% accuracy")
print("    but catches ZERO churners. Must use class weights or SMOTE.")


## 5. Missing Value Check

Missing values must be identified before training — neural networks cannot process `NaN` inputs.


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
})

total_missing = missing.sum()
print(f"Total missing cells: {total_missing}")
print()
if total_missing == 0:
    print("✅ No missing values in any column — dataset is fully complete.")
    print("   No imputation needed before training.")
else:
    print(missing_df[missing_df['Missing Count'] > 0])


## 6. Basic Statistical Summary

We extend `describe()` with the **Coefficient of Variation (CV%)** = `std / |mean| × 100`.

A high CV% (> 50%) means the feature has very high relative spread — these features can cause
**gradient explosion** during backpropagation if not standardised before training.


In [ ]:
desc = df[num_cols].describe().T.round(3)
desc.columns = ['count','mean','std','min','25%','50%','75%','max']
desc['cv%'] = (desc['std'] / desc['mean'].abs() * 100).round(1)
desc['corr_churn'] = df[num_cols].corrwith(df['churn']).round(4).values
desc['scale?'] = desc['cv%'].apply(lambda x: '⚠ YES' if x > 30 else 'no')

print("Statistical Summary of Numerical Features:")
print(desc[['mean','std','min','max','cv%','corr_churn','scale?']].to_string())
print()
print("Features requiring standardisation (CV% > 30):")
print([c for c in num_cols if (df[c].std()/abs(df[c].mean())*100) > 30])


### Categorical Feature Breakdown

In [ ]:
for col in cat_cols:
    print(f"\n{col.upper()}")
    vc = df[col].value_counts()
    for cat, cnt in vc.items():
        print(f"  {cat:<25} {cnt:>4}  ({cnt/len(df)*100:.1f}%)")


## 7. Distribution of the Target Variable

Visual exploration of the `churn` column — class counts, proportions, and imbalance summary.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Target Variable Distribution: churn", fontsize=14, fontweight="bold")

vc = df['churn'].value_counts().sort_index()
labels = ["Retained (0)", "Churned (1)"]
colors = [BLUE, RED]

# Bar chart
bars = axes[0].bar(labels, vc.values, color=colors, edgecolor="white", linewidth=1.5, width=0.5)
for bar, val in zip(bars, vc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=11, fontweight="bold")
axes[0].set_title("Class Counts", fontweight="bold")
axes[0].set_ylabel("Number of Customers")
axes[0].set_ylim(0, 2300)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    vc.values, labels=labels, colors=colors, autopct="%1.1f%%", startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2))
for at in autotexts:
    at.set_fontweight("bold"); at.set_fontsize(12)
axes[1].set_title("Class Proportions", fontweight="bold")

# Info panel
axes[2].axis("off")
imbalance = vc[0] / vc[1]
info = (f"Dataset Size  : 2,000\n\n"
        f"Retained (0)  : 1,969  (98.4%)\n"
        f"Churned  (1)  :    31   (1.6%)\n\n"
        f"Imbalance Ratio:\n"
        f"  {imbalance:.1f} : 1\n\n"
        f"⚠  Highly imbalanced!\n"
        f"   Use pos_weight in BCELoss\n"
        f"   or apply SMOTE.")
axes[2].text(0.05, 0.95, info, transform=axes[2].transAxes, va="top",
             fontsize=11, fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.6", facecolor="white",
                       edgecolor=RED, linewidth=1.5))
axes[2].set_title("Imbalance Summary", fontweight="bold")

plt.tight_layout()
plt.show()
print("Observation: Extreme 63.5:1 class imbalance — standard accuracy is misleading.")
print("Use precision, recall, F1-score, and AUC-ROC as evaluation metrics.")


## 8. Numerical Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Numerical Feature Distributions (Histogram + KDE)", fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color=BLUE, bins=28,
                 edgecolor="white", linewidth=0.3, alpha=0.8)
    axes[i].axvline(df[col].mean(), color=RED, linestyle="--",
                    linewidth=1.4, label=f"μ={df[col].mean():.1f}")
    axes[i].set_title(col.replace("_"," ").title(), fontsize=9, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].legend(fontsize=7)

plt.tight_layout()
plt.show()
print("Observation: Most features are right-skewed. payment_delay_days and")
print("last_complaint_days_ago have long tails — standardisation is essential.")


## 9. Categorical Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Categorical Feature Distributions", fontsize=14, fontweight="bold")
axes = axes.flatten()
palette = sns.color_palette("Blues_d", 6)

for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    bars = axes[i].barh(vc.index, vc.values, color=palette[:len(vc)], edgecolor="white")
    for bar, val in zip(bars, vc.values):
        axes[i].text(val + 5, bar.get_y() + bar.get_height()/2,
                     f"{val} ({val/len(df)*100:.1f}%)", va="center", fontsize=9)
    axes[i].set_title(col.replace("_"," ").title(), fontweight="bold")
    axes[i].set_xlabel("Count")

plt.tight_layout()
plt.show()
print("Observation: contract_type is dominated by Month-to-month (55.6%) — a known")
print("churn-risk segment. All 4 columns need one-hot encoding before training.")


## 10. Correlation Heatmap

In [ ]:
corr_cols = num_cols + ['autopay_enabled', 'churn']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 8},
            cbar_kws={"shrink": 0.8})
ax.set_title("Pearson Correlation Heatmap (lower triangle)", fontsize=13,
             fontweight="bold", pad=12)
plt.tight_layout()
plt.show()
print("Observation: support_tickets (r=+0.12) and satisfaction_score (r=-0.09)")
print("show the strongest correlations with churn — consistent with domain knowledge.")
print("Low inter-feature correlation means multicollinearity is not a concern.")


## 11. Feature Behaviour by Churn Class (Box Plots)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Numerical Features — Retained vs Churned", fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data0 = df[df['churn']==0][col]
    data1 = df[df['churn']==1][col]
    bp = axes[i].boxplot([data0, data1], patch_artist=True,
                          tick_labels=["Retained","Churned"],
                          medianprops=dict(color="black", linewidth=2))
    bp['boxes'][0].set_facecolor(BLUE + "99")
    bp['boxes'][1].set_facecolor(RED  + "99")
    axes[i].set_title(col.replace("_"," ").title(), fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()
print("Observation: Churned customers tend to have higher support_tickets,")
print("higher payment_delay_days, and lower satisfaction_score.")


## 12. Save Results to `results/` Folder

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving

# ── evaluation_outputs.png ──────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.suptitle("Task 1 – Dataset Evaluation Outputs\nCustomer Churn Neural Network Dataset",
             fontsize=16, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.52, wspace=0.38)

# Panel A: target bar
ax1 = fig.add_subplot(gs[0, 0])
vc = df['churn'].value_counts().sort_index()
bars = ax1.bar(["Retained (0)","Churned (1)"], vc.values,
               color=[BLUE, RED], edgecolor="white", linewidth=1.5, width=0.5)
for bar, val in zip(bars, vc.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
             f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center",
             fontsize=10, fontweight="bold")
ax1.set_title("A. Target Class Distribution", fontweight="bold", fontsize=11)
ax1.set_ylim(0, 2300); ax1.set_ylabel("Count")

# Panel B: pie
ax2 = fig.add_subplot(gs[0, 1])
wedges, texts, autotexts = ax2.pie(vc.values, labels=["Retained","Churned"],
    colors=[BLUE, RED], autopct="%1.1f%%", startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2))
for at in autotexts: at.set_fontweight("bold"); at.set_fontsize(11)
ax2.set_title("B. Churn Proportion", fontweight="bold", fontsize=11)

# Panel C: missing values
ax3 = fig.add_subplot(gs[0, 2])
missing = df.isnull().sum()
colors_mv = [RED if v > 0 else "#10B981" for v in missing.values]
ax3.barh(missing.index, missing.values, color=colors_mv, edgecolor="white")
ax3.set_title("C. Missing Values per Column", fontweight="bold", fontsize=11)
ax3.set_xlabel("Missing Count")
green_p = mpatches.Patch(color="#10B981", label="0 missing ✓")
ax3.legend(handles=[green_p], fontsize=8)

# Panels D-H: numerical histograms
highlight = ['tenure_months','monthly_charges_inr','satisfaction_score',
             'payment_delay_days','support_tickets_last_90_days']
htitles   = ['D. Tenure (months)','E. Monthly Charges (INR)',
             'F. Satisfaction Score','G. Payment Delay (days)',
             'H. Support Tickets (90d)']
positions = [(1,0),(1,1),(1,2),(2,0),(2,1)]

for (r,c), col, title in zip(positions, highlight, htitles):
    ax = fig.add_subplot(gs[r, c])
    sns.histplot(df[col], kde=True, ax=ax, color=BLUE, bins=28,
                 edgecolor="white", linewidth=0.3, alpha=0.8)
    ax.axvline(df[col].mean(), color=RED, linestyle="--", linewidth=1.4,
               label=f"μ={df[col].mean():.1f}")
    ax.set_title(title, fontweight="bold", fontsize=9)
    ax.set_xlabel(""); ax.legend(fontsize=8)

# Panel I: correlation with churn
ax_c = fig.add_subplot(gs[2, 2])
corr_vals = df[num_cols+['autopay_enabled']].corrwith(df['churn']).sort_values()
ax_c.barh(corr_vals.index, corr_vals.values,
          color=[RED if v>0 else BLUE for v in corr_vals.values], edgecolor="white")
ax_c.axvline(0, color="black", linewidth=0.8)
ax_c.set_title("I. Feature Correlation\nwith Churn", fontweight="bold", fontsize=9)
ax_c.set_xlabel("Pearson r")

plt.savefig(RESULTS_DIR / "evaluation_outputs.png", bbox_inches="tight")
plt.close()
print("✓ results/evaluation_outputs.png saved")

# ── model_comparison_table.csv ──────────────────────────────────────
desc = df[num_cols].describe().T.round(3)
desc.columns = ['count','mean','std','min','25%','50%','75%','max']
desc['cv_percent']       = (desc['std'] / desc['mean'].abs() * 100).round(1)
desc['missing']          = df[num_cols].isnull().sum().values
desc['corr_with_churn']  = df[num_cols].corrwith(df['churn']).round(4).values
desc['scaling_needed']   = desc['cv_percent'].apply(lambda x: 'Yes' if x > 30 else 'No')
desc.index.name = 'feature'
desc.to_csv(RESULTS_DIR / "model_comparison_table.csv")
print("✓ results/model_comparison_table.csv saved")
print()
print("All results saved to results/ folder.")


## 13. Task 1 Summary & Key Takeaways

| Check | Finding |
|-------|---------|
| **Shape** | 2,000 rows × 17 columns |
| **Numerical features** | 10 |
| **Categorical features** | 4 (`region`, `plan_type`, `contract_type`, `payment_method`) |
| **Binary features** | 1 (`autopay_enabled`) |
| **Identifier** | `customer_id` — dropped before training |
| **Missing values** | **None** — dataset is fully complete |
| **Target** | `churn` — binary (0/1) — Binary Classification |
| **Class imbalance** | 63.5 : 1 — **severe imbalance** |
| **Scaling needed** | 8 of 10 numerical features (CV% > 30) |

### Implications for Neural Network Design (Task 2)
- **Drop** `customer_id` before building input tensors
- **One-hot encode** 4 categorical features → expands input dimensionality
- **Z-score standardise** all numerical features (especially high-CV% ones)
- Use `BCEWithLogitsLoss(pos_weight=torch.tensor([63.5]))` to handle class imbalance
- Evaluation metrics: **Precision, Recall, F1-score, AUC-ROC** (not accuracy)
